<a href="https://colab.research.google.com/github/batinylmz/financialAnomalyDetection/blob/main/1D_CNNAutoEncoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import layers

# 1. Veriyi Oku ve Temizle
df = pd.read_csv("/content/drive/MyDrive/financial_anomaly_benchmark_data.csv")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# 2. Gerçekçi Anomali Etiketi Oluşturma (En yüksek %1'lik çarpım değeri)
carpim = df['Volume_Change'] * df['Volatility_HighLow']
threshold_val = np.percentile(carpim, 99) # Çarpımın en yüksek %1'lik sınırını bul

# Eğer çarpım bu sınırdan büyükse 1 (Anomali), değilse 0 (Normal)
y_true = (carpim > threshold_val).astype(int)

print(f"Toplam Veri: {len(y_true)} | Bulunan Anomali Noktası: {sum(y_true)}")

# 3. Özellikleri Seçme (Veri sızıntısını önlemek için o 2 sütunu sildik!)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Returns']
data = df[features].values

# 4. Ölçeklendirme (Sadece kalan 6 sütun ölçeklendiriliyor)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# 5. Zaman Serisi Pencereleri Oluşturma
SEQ_LEN = 24
def create_sequences(data, labels, seq_length):
    X, Y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        # Pencere içinde 1 tane bile anomali varsa, tüm pencere anomali alarmı verir
        if np.sum(labels.iloc[i : i + seq_length] if isinstance(labels, pd.Series) else labels[i : i + seq_length]) > 0:
            Y.append(1)
        else:
            Y.append(0)
    return np.array(X), np.array(Y)

X_seq, y_seq = create_sequences(data_scaled, y_true, SEQ_LEN)

# Veriyi Eğitim (%70) ve Test (%30) olarak ayır
split_idx = int(len(X_seq) * 0.7)
X_train, y_train = X_seq[:split_idx], y_seq[:split_idx]
X_test, y_test = X_seq[split_idx:], y_seq[split_idx:]

# Modeller SADECE normal verilerle eğitilir
X_train_normal = X_train[y_train == 0]

print(f"Eğitime Giren Özellik (Sütun) Sayısı: {len(features)}")
print(f"Eğitim Verisi Şekli: {X_train_normal.shape}")
print(f"Test Verisi Şekli: {X_test.shape}")
print(f"Test Setindeki Anormal Pencere Sayısı: {sum(y_test)}")

Toplam Veri: 141004 | Bulunan Anomali Noktası: 1411
Eğitime Giren Özellik (Sütun) Sayısı: 6
Eğitim Verisi Şekli: (76661, 24, 6)
Test Verisi Şekli: (42294, 24, 6)
Test Setindeki Anormal Pencere Sayısı: 6948


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. 1D-CNN Autoencoder Çekirdek Mimarisi
inputs = layers.Input(shape=(SEQ_LEN, len(features)))
x = layers.Conv1D(16, 3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(2, padding='same')(x)
x = layers.Conv1D(8, 3, activation='relu', padding='same')(x)
x = layers.UpSampling1D(2)(x)
x = layers.Conv1D(16, 3, activation='relu', padding='same')(x)
outputs = layers.Conv1D(len(features), 3, activation='linear', padding='same')(x)
base_cnn = Model(inputs, outputs)

# 2. Özel Geri Yayılım ve Ceza Sistemi Sınıfı (Contrastive Loss)
class ContrastiveCNN(Model):
    def __init__(self, base_model, margin=3.0, anomaly_weight=15.0, **kwargs):
        super().__init__(**kwargs)
        self.base_model = base_model
        self.margin = margin                  # Anomali hatasının itileceği alt sınır
        self.anomaly_weight = anomaly_weight  # Azınlık sınıfı için CEZA ÇARPANI!

    def compile(self, optimizer, **kwargs):
        super().compile(**kwargs)
        # Ensure the optimizer is an actual object, not just a string name
        if isinstance(optimizer, str):
            self.optimizer = tf.keras.optimizers.get(optimizer)
        else:
            self.optimizer = optimizer

    def train_step(self, data):
        x, y = data # x: zaman serisi penceresi, y: anomali etiketi (0 veya 1)
        y = tf.cast(y, tf.float32)

        with tf.GradientTape() as tape:
            x_pred = self.base_model(x, training=True)
            # Her pencere için MSE hatası
            mse = tf.reduce_mean(tf.square(x - x_pred), axis=[1, 2])

            # NORMAL DURUMLAR (y=0): Sadece hatayı küçültmeye çalış
            loss_normal = (1.0 - y) * mse

            # ANORMAL DURUMLAR (y=1): Eğer MSE margin'den (3.0) küçükse, modeli şiddetle cezalandır!
            # Bunu anomaly_weight ile çarparak azınlık sınıfı dengesizliğini çözüyoruz.
            loss_anomaly = y * tf.maximum(0.0, self.margin - mse) * self.anomaly_weight

            total_loss = tf.reduce_mean(loss_normal + loss_anomaly)

        # Geri Yayılım (Backpropagation)
        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return {"loss": total_loss}

    def call(self, inputs):
        return self.base_model(inputs)

# 3. Eğitimi Başlatma
# DİKKAT: Artık modeli "sadece normal verilerle" değil, CEZA uygulayabilmek için "TÜM veriler (X_train, y_train)" ile eğitiyoruz!
cnn_contrastive = ContrastiveCNN(base_cnn, margin=3.0, anomaly_weight=15.0)
cnn_contrastive.compile(optimizer='adam')

print("Contrastive 1D-CNN Eğitiliyor (Anomaliler ağır cezalandırılıyor)...")
history_cnn = cnn_contrastive.fit(X_train, y_train, epochs=15, batch_size=64, verbose=1)

# 4. Tahmin ve Değerlendirme
cnn_pred = cnn_contrastive.predict(X_test)
cnn_mse = np.mean(np.power(X_test - cnn_pred, 2), axis=(1, 2))

# Eğitimdeki hatalara göre bir eşik belirliyoruz. Ceza uyguladığımız için ayırım artık daha keskin olacak.
threshold_cnn = np.percentile(cnn_mse, 85)
y_pred_cnn = (cnn_mse > threshold_cnn).astype(int)

print("\n--- CONTRASTIVE 1D-CNN SONUÇLARI ---")
print(f"Accuracy (Doğruluk) : {accuracy_score(y_test, y_pred_cnn):.4f}")
print(f"Precision (Kesinlik): {precision_score(y_test, y_pred_cnn):.4f}")
print(f"Recall (Duyarlılık) : {recall_score(y_test, y_pred_cnn):.4f}")
print(f"F1 Score            : {f1_score(y_test, y_pred_cnn):.4f}")

Contrastive 1D-CNN Eğitiliyor (Anomaliler ağır cezalandırılıyor)...
Epoch 1/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0000e+00
Epoch 2/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0000e+00
Epoch 3/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0000e+00
Epoch 4/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0000e+00
Epoch 5/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0000e+00
Epoch 6/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0000e+00
Epoch 7/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0000e+00
Epoch 8/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0000e+00
Epoch 9/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0000e+00
Epoch 10/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0000e+00
Epoch 11/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0000e+00
Epoch 12/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0000e+00
Epoch 13/15
1542/1542 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/ste